# Bias Correction para Datos SSP - ACCESS-CM2

Este notebook implementa el pipeline completo de bias correction para escenarios SSP usando quantile mapping (DQM/EQM) aplicado a datos históricos ya entrenados.

## Región: Valle de Aconcagua, Chile
- **Variables**: pr, tasmin, tasmax
- **Escenarios**: SSP245, SSP370, SSP585
- **Período**: 2015-2100
- **Método**: Detrended Quantile Mapping (DQM) con parámetros pre-entrenados
- **Resolución final**: 0.05° (rejilla CR2MET)

## Pipeline de Procesamiento:
1. **Cargar y concatenar** archivos temporales SSP
2. **Convertir unidades** (pr: kg m-2 s-1 → mm/day, tas: K → °C)
3. **Regridding** a resolución CR2MET (0.05°)
4. **Recorte espacial** a Valle de Aconcagua
5. **Bias correction** usando parámetros del historical
6. **Exportar** NetCDF corregidos

In [1]:
# ============================================================
# 🚀 SECCIÓN 1: SETUP Y CONFIGURACIÓN
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import time
from datetime import datetime
import sys
import pickle
import json
import logging

# Configurar warnings
warnings.filterwarnings('ignore')

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("=" * 70)
print("🎯 SSP BIAS CORRECTION PIPELINE - ACCESS-CM2")
print("=" * 70)
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"xarray version: {xr.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

🎯 SSP BIAS CORRECTION PIPELINE - ACCESS-CM2
Inicio: 2025-10-12 21:09:51
xarray version: 2025.1.2
pandas version: 2.3.3
numpy version: 2.3.3


In [2]:
# ============================================================
# 📁 CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# ============================================================

# Definir rutas principales
base_dir = Path("/home/aninotna/magister/tesis/justh2_pipeline")
data_dir = base_dir / "data"
cmip6_data_dir = data_dir / "cmip6"
cr2met_path = data_dir / "cr2met/clima.zarr"

# Rutas de datos SSP por variable
ssp_data_paths = {
    'pr': cmip6_data_dir / "pr",
    'tasmin': cmip6_data_dir / "tmin", 
    'tasmax': cmip6_data_dir / "tmax"
}

# Directorios de salida
out_dir = base_dir / "out"
bias_params_dir = out_dir / "bias_params/ACCESS-CM2"  # Parámetros ya entrenados
corrected_ssp_dir = out_dir / "corrected_ssp/ACCESS-CM2"
logs_dir = base_dir / "logs"

# Crear directorios de salida
for directory in [corrected_ssp_dir, logs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("📁 CONFIGURACIÓN DE RUTAS:")
print(f"  • Base: {base_dir}")
print(f"  • CR2MET: {cr2met_path}")
print(f"  • CMIP6 data: {cmip6_data_dir}")
print(f"  • Bias params (entrenados): {bias_params_dir}")
print(f"  • Corrected SSP output: {corrected_ssp_dir}")
print(f"  • Logs: {logs_dir}")

# Verificar que los directorios críticos existen
print(f"\n🔍 VERIFICACIÓN:")
print(f"  CR2MET existe: {'✅' if cr2met_path.exists() else '❌'}")
print(f"  CMIP6 data existe: {'✅' if cmip6_data_dir.exists() else '❌'}")
print(f"  Bias params existe: {'✅' if bias_params_dir.exists() else '❌'}")

for var, path in ssp_data_paths.items():
    exists = '✅' if path.exists() else '❌'
    print(f"  {var} SSP data: {exists}")

if not bias_params_dir.exists():
    print(f"\n⚠️ ADVERTENCIA: No se encontraron parámetros de bias correction entrenados")
    print(f"   Necesitarás ejecutar primero el notebook de historical bias correction")

📁 CONFIGURACIÓN DE RUTAS:
  • Base: /home/aninotna/magister/tesis/justh2_pipeline
  • CR2MET: /home/aninotna/magister/tesis/justh2_pipeline/data/cr2met/clima.zarr
  • CMIP6 data: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6
  • Bias params (entrenados): /home/aninotna/magister/tesis/justh2_pipeline/out/bias_params/ACCESS-CM2
  • Corrected SSP output: /home/aninotna/magister/tesis/justh2_pipeline/out/corrected_ssp/ACCESS-CM2
  • Logs: /home/aninotna/magister/tesis/justh2_pipeline/logs

🔍 VERIFICACIÓN:
  CR2MET existe: ✅
  CMIP6 data existe: ✅
  Bias params existe: ✅
  pr SSP data: ✅
  tasmin SSP data: ✅
  tasmax SSP data: ✅


In [3]:
# ============================================================
# ⚙️ CONFIGURACIÓN DE PARÁMETROS DEL PROCESAMIENTO
# ============================================================

# Configuración de escenarios y variables
ssp_scenarios = ['ssp245', 'ssp370', 'ssp585']
variables = ['pr', 'tasmin', 'tasmax']
model_name = 'ACCESS-CM2'

# Configuración región Valle de Aconcagua
valle_aconcagua_bounds = {
    'lat_min': -33.27,
    'lat_max': -32.26,
    'lon_min': -71.89,
    'lon_max': -70.00
}

# Configuración de conversión de unidades
unit_conversions = {
    'pr': {
        'from': 'kg m-2 s-1',
        'to': 'mm/day',
        'factor': 86400,  # kg m-2 s-1 * 86400 s/day = mm/day
        'description': 'Precipitación: kg m-2 s-1 → mm/day'
    },
    'tasmin': {
        'from': 'K',
        'to': '°C',
        'factor': -273.15,  # K - 273.15 = °C
        'description': 'Temperatura mínima: K → °C'
    },
    'tasmax': {
        'from': 'K', 
        'to': '°C',
        'factor': -273.15,  # K - 273.15 = °C
        'description': 'Temperatura máxima: K → °C'
    }
}

print("⚙️ CONFIGURACIÓN DEL PROCESAMIENTO:")
print(f"📊 Modelo: {model_name}")
print(f"🔮 Escenarios: {ssp_scenarios}")
print(f"📈 Variables: {variables}")

print(f"\n📍 Región Valle de Aconcagua:")
for key, value in valle_aconcagua_bounds.items():
    print(f"    {key}: {value}")

print(f"\n🔧 Conversiones de unidades:")
for var, config in unit_conversions.items():
    print(f"  {var}: {config['description']}")

⚙️ CONFIGURACIÓN DEL PROCESAMIENTO:
📊 Modelo: ACCESS-CM2
🔮 Escenarios: ['ssp245', 'ssp370', 'ssp585']
📈 Variables: ['pr', 'tasmin', 'tasmax']

📍 Región Valle de Aconcagua:
    lat_min: -33.27
    lat_max: -32.26
    lon_min: -71.89
    lon_max: -70.0

🔧 Conversiones de unidades:
  pr: Precipitación: kg m-2 s-1 → mm/day
  tasmin: Temperatura mínima: K → °C
  tasmax: Temperatura máxima: K → °C


## 🔍 SECCIÓN 2: EXPLORACIÓN Y ANÁLISIS DE ARCHIVOS SSP

Antes de procesar los datos, necesitamos:
1. **Descubrir** qué archivos SSP están disponibles
2. **Inspeccionar** estructura, dimensiones y metadatos
3. **Verificar** rangos temporales y continuidad
4. **Analizar** coordenadas espaciales y unidades
5. **Planificar** la concatenación temporal

In [4]:
# ============================================================
# 📂 DESCUBRIR ARCHIVOS SSP DISPONIBLES
# ============================================================

def discover_ssp_files():
    """Descubre todos los archivos SSP disponibles por variable y escenario"""
    available_files = {}
    
    print("=" * 70)
    print("📂 EXPLORANDO ARCHIVOS SSP DISPONIBLES")
    print("=" * 70)
    
    for var, var_path in ssp_data_paths.items():
        print(f"\n--- Variable: {var.upper()} ---")
        print(f"Directorio: {var_path}")
        
        if not var_path.exists():
            print(f"  ❌ Directorio no encontrado")
            continue
            
        available_files[var] = {}
        
        for scenario in ssp_scenarios:
            scenario_path = var_path / scenario
            print(f"\n  📁 Escenario: {scenario.upper()}")
            
            if not scenario_path.exists():
                print(f"    ❌ Directorio {scenario} no encontrado")
                continue
                
            # Buscar archivos ACCESS-CM2 (archivos .nc que contengan ACCESS-CM2)
            nc_files = list(scenario_path.glob(f"*{model_name}*.nc"))
            
            if nc_files:
                available_files[var][scenario] = sorted(nc_files)
                print(f"    ✅ Archivos encontrados: {len(nc_files)}")
                for file in nc_files:
                    print(f"      📄 {file.name}")
            else:
                print(f"    ❌ No se encontraron archivos {model_name}")
                
                # Mostrar todos los archivos disponibles para debug
                all_files = list(scenario_path.glob("*.nc"))
                if all_files:
                    print(f"    🔍 Archivos disponibles:")
                    for file in all_files[:3]:  # Mostrar solo los primeros 3
                        print(f"      📄 {file.name}")
                    if len(all_files) > 3:
                        print(f"      ... y {len(all_files)-3} más")
    
    return available_files

# Descubrir archivos
available_files = discover_ssp_files()

# Resumen
print(f"\n{'='*70}")
print(f"📊 RESUMEN DE ARCHIVOS SSP DISPONIBLES")
print(f"{'='*70}")

total_files = 0
for var in available_files:
    var_total = sum(len(files) for files in available_files[var].values())
    total_files += var_total
    print(f"\n{var.upper()}: {var_total} archivos")
    for scenario in available_files[var]:
        print(f"  {scenario}: {len(available_files[var][scenario])} archivos")

print(f"\n📦 TOTAL: {total_files} archivos SSP encontrados")
print(f"✅ Cobertura: {len(variables)} variables × {len(ssp_scenarios)} escenarios")
print(f"📅 Archivos típicamente divididos en 2 períodos temporales")
print(f"{'='*70}")

📂 EXPLORANDO ARCHIVOS SSP DISPONIBLES

--- Variable: PR ---
Directorio: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/pr

  📁 Escenario: SSP245
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP370
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP585
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp585_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp585_r1i1p1f1_gn_20640706-21001231.nc

--- Variable: TASMIN ---
Directorio: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/tmin

  📁 Escenario: SSP245
    ✅ Archivos encontrados: 2
      📄 tasmin_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
      📄 tasmin_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP3

In [5]:
# ============================================================
# 🔍 INSPECCIÓN DETALLADA DE ARCHIVOS SSP
# ============================================================

def inspect_ssp_file_structure():
    """Inspecciona la estructura de un archivo representativo de cada variable"""
    
    print("=" * 70)
    print("🔍 INSPECCIÓN DE ESTRUCTURA DE ARCHIVOS SSP")
    print("=" * 70)
    
    file_info = {}
    
    for var in variables:
        if var in available_files and 'ssp245' in available_files[var]:
            # Tomar el primer archivo de ssp245 como muestra
            sample_file = available_files[var]['ssp245'][0]
            
            print(f"\n--- Analizando {var.upper()} ---")
            print(f"📄 Archivo: {sample_file.name}")
            
            try:
                # Abrir archivo para inspección
                with xr.open_dataset(sample_file) as ds:
                    print(f"\n📊 Dimensiones:")
                    for dim_name, dim_size in ds.dims.items():
                        print(f"  {dim_name}: {dim_size}")
                    
                    print(f"\n📈 Variables de datos:")
                    for var_name in ds.data_vars:
                        var_obj = ds[var_name]
                        print(f"  {var_name}:")
                        print(f"    Dimensiones: {var_obj.dims}")
                        print(f"    Forma: {var_obj.shape}")
                        if hasattr(var_obj, 'units'):
                            print(f"    Unidades: {var_obj.units}")
                        if hasattr(var_obj, 'long_name'):
                            print(f"    Nombre: {var_obj.long_name}")
                    
                    print(f"\n🌍 Coordenadas espaciales:")
                    if 'lat' in ds.coords:
                        lat_range = f"{ds.lat.min().values:.3f} a {ds.lat.max().values:.3f}"
                        print(f"  Latitud: {lat_range} ({len(ds.lat)} puntos)")
                    if 'lon' in ds.coords:
                        lon_range = f"{ds.lon.min().values:.3f} a {ds.lon.max().values:.3f}"
                        print(f"  Longitud: {lon_range} ({len(ds.lon)} puntos)")
                    
                    print(f"\n📅 Información temporal:")
                    if 'time' in ds.coords:
                        time_start = pd.to_datetime(ds.time.values[0])
                        time_end = pd.to_datetime(ds.time.values[-1])
                        print(f"  Inicio: {time_start.strftime('%Y-%m-%d')}")
                        print(f"  Final: {time_end.strftime('%Y-%m-%d')}")
                        print(f"  Total días: {len(ds.time)}")
                        
                        # Verificar calendario
                        if hasattr(ds.time, 'calendar'):
                            print(f"  Calendario: {ds.time.calendar}")
                    
                    # Guardar info para comparación
                    file_info[var] = {
                        'dims': dict(ds.dims),
                        'temporal_range': (time_start, time_end),
                        'spatial_extent': {
                            'lat_min': float(ds.lat.min()),
                            'lat_max': float(ds.lat.max()),
                            'lon_min': float(ds.lon.min()),
                            'lon_max': float(ds.lon.max())
                        }
                    }
                    
            except Exception as e:
                print(f"  ❌ Error al inspeccionar archivo: {e}")
                file_info[var] = {'error': str(e)}
    
    return file_info

# Inspeccionar archivos
file_info = inspect_ssp_file_structure()

# Verificar consistencia entre variables
print(f"\n{'='*70}")
print(f"✅ VERIFICACIÓN DE CONSISTENCIA")
print(f"{'='*70}")

# Verificar que todas las variables tengan las mismas dimensiones espaciales
spatial_dims = {}
for var, info in file_info.items():
    if 'dims' in info:
        spatial_dims[var] = {k: v for k, v in info['dims'].items() if k in ['lat', 'lon']}

if len(set(str(dims) for dims in spatial_dims.values())) == 1:
    print("✅ Todas las variables tienen dimensiones espaciales consistentes")
    ref_dims = list(spatial_dims.values())[0]
    print(f"   Resolución espacial: {ref_dims}")
else:
    print("⚠️ Las variables tienen dimensiones espaciales diferentes:")
    for var, dims in spatial_dims.items():
        print(f"   {var}: {dims}")

print(f"\n🌍 Cobertura espacial Valle de Aconcagua:")
valle_covered = True
for var, info in file_info.items():
    if 'spatial_extent' in info:
        extent = info['spatial_extent']
        lat_ok = (extent['lat_min'] <= valle_aconcagua_bounds['lat_min'] and 
                  extent['lat_max'] >= valle_aconcagua_bounds['lat_max'])
        lon_ok = (extent['lon_min'] <= valle_aconcagua_bounds['lon_min'] and 
                  extent['lon_max'] >= valle_aconcagua_bounds['lon_max'])
        
        status = "✅" if (lat_ok and lon_ok) else "⚠️"
        print(f"   {var}: {status}")
        if not (lat_ok and lon_ok):
            valle_covered = False

if valle_covered:
    print("✅ Todos los archivos cubren completamente el Valle de Aconcagua")
else:
    print("⚠️ Algunos archivos podrían no cubrir completamente la región")

print(f"{'='*70}")

🔍 INSPECCIÓN DE ESTRUCTURA DE ARCHIVOS SSP

--- Analizando PR ---
📄 Archivo: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc

📊 Dimensiones:
  time: 18084
  bnds: 2
  lat: 144
  lon: 192

📈 Variables de datos:
  time_bnds:
    Dimensiones: ('time', 'bnds')
    Forma: (18084, 2)
  lat_bnds:
    Dimensiones: ('time', 'lat', 'bnds')
    Forma: (18084, 144, 2)
  lon_bnds:
    Dimensiones: ('time', 'lon', 'bnds')
    Forma: (18084, 192, 2)
  pr:
    Dimensiones: ('time', 'lat', 'lon')
    Forma: (18084, 144, 192)
    Unidades: kg m-2 s-1
    Nombre: Precipitation

🌍 Coordenadas espaciales:
  Latitud: -89.375 a 89.375 (144 puntos)
  Longitud: 0.938 a 359.062 (192 puntos)

📅 Información temporal:
  Inicio: 2015-01-01
  Final: 2064-07-05
  Total días: 18084

--- Analizando TASMIN ---
📄 Archivo: tasmin_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc

📊 Dimensiones:
  time: 18084
  bnds: 2
  lat: 144
  lon: 192

📈 Variables de datos:
  time_bnds:
    Dimensiones: ('time', 'bnds'

In [6]:
# ============================================================
# 📅 VERIFICACIÓN DE CONTINUIDAD TEMPORAL ENTRE PERÍODOS
# ============================================================

def verify_temporal_continuity():
    """Verifica que los dos períodos temporales de cada archivo se conecten correctamente"""
    
    print("=" * 70)
    print("📅 VERIFICACIÓN DE CONTINUIDAD TEMPORAL")
    print("=" * 70)
    
    temporal_analysis = {}
    
    for var in variables:
        print(f"\n--- Variable: {var.upper()} ---")
        
        if var not in available_files:
            print(f"  ❌ Variable {var} no disponible")
            continue
            
        temporal_analysis[var] = {}
        
        for scenario in ssp_scenarios:
            print(f"\n  📁 Escenario: {scenario.upper()}")
            
            if scenario not in available_files[var]:
                print(f"    ❌ Escenario {scenario} no disponible")
                continue
                
            files = available_files[var][scenario]
            
            if len(files) != 2:
                print(f"    ⚠️ Se esperaban 2 archivos, encontrados: {len(files)}")
                continue
            
            # Ordenar archivos por fecha de inicio (debería estar en el nombre)
            files = sorted(files)
            
            try:
                # Analizar primer archivo (2015-2064)
                with xr.open_dataset(files[0]) as ds1:
                    start1 = pd.to_datetime(ds1.time.values[0])
                    end1 = pd.to_datetime(ds1.time.values[-1])
                    n_days1 = len(ds1.time)
                
                # Analizar segundo archivo (2064-2100)
                with xr.open_dataset(files[1]) as ds2:
                    start2 = pd.to_datetime(ds2.time.values[0])
                    end2 = pd.to_datetime(ds2.time.values[-1])
                    n_days2 = len(ds2.time)
                
                # Verificar continuidad
                gap_days = (start2 - end1).days
                overlap_days = max(0, (end1 - start2).days + 1)
                
                print(f"    📄 Archivo 1: {start1.strftime('%Y-%m-%d')} a {end1.strftime('%Y-%m-%d')} ({n_days1:,} días)")
                print(f"    📄 Archivo 2: {start2.strftime('%Y-%m-%d')} a {end2.strftime('%Y-%m-%d')} ({n_days2:,} días)")
                
                if gap_days == 1:
                    print(f"    ✅ Continuidad perfecta (1 día de diferencia)")
                    status = "perfect"
                elif gap_days == 0:
                    print(f"    ✅ Sin gap temporal")
                    status = "no_gap"
                elif overlap_days > 0:
                    print(f"    ⚠️ Solapamiento de {overlap_days} días")
                    status = "overlap"
                elif gap_days > 1:
                    print(f"    ❌ Gap de {gap_days} días")
                    status = "gap"
                else:
                    print(f"    ✅ Archivos consecutivos")
                    status = "consecutive"
                
                # Calcular período total
                total_days = n_days1 + n_days2 - max(0, overlap_days)
                total_years = total_days / 365.25
                
                print(f"    📊 Período completo: {start1.strftime('%Y-%m-%d')} a {end2.strftime('%Y-%m-%d')}")
                print(f"    📈 Total: {total_days:,} días (~{total_years:.1f} años)")
                
                temporal_analysis[var][scenario] = {
                    'file1': {'start': start1, 'end': end1, 'days': n_days1},
                    'file2': {'start': start2, 'end': end2, 'days': n_days2},
                    'gap_days': gap_days,
                    'overlap_days': overlap_days,
                    'total_days': total_days,
                    'total_years': total_years,
                    'status': status
                }
                
            except Exception as e:
                print(f"    ❌ Error al analizar archivos: {e}")
                temporal_analysis[var][scenario] = {'error': str(e)}
    
    return temporal_analysis

# Verificar continuidad temporal
temporal_analysis = verify_temporal_continuity()

# Resumen de continuidad
print(f"\n{'='*70}")
print(f"📊 RESUMEN DE CONTINUIDAD TEMPORAL")
print(f"{'='*70}")

status_counts = {'perfect': 0, 'no_gap': 0, 'consecutive': 0, 'overlap': 0, 'gap': 0, 'error': 0}

for var in temporal_analysis:
    print(f"\n{var.upper()}:")
    for scenario in temporal_analysis[var]:
        analysis = temporal_analysis[var][scenario]
        if 'error' in analysis:
            print(f"  {scenario}: ❌ Error")
            status_counts['error'] += 1
        else:
            status = analysis['status']
            status_counts[status] += 1
            years = analysis['total_years']
            print(f"  {scenario}: ✅ {status} ({years:.1f} años)")

print(f"\n📈 Estado general:")
for status, count in status_counts.items():
    if count > 0:
        emoji = "✅" if status in ['perfect', 'no_gap', 'consecutive'] else "⚠️" if status == 'overlap' else "❌"
        print(f"  {emoji} {status}: {count} casos")

total_combinations = len(variables) * len(ssp_scenarios)
successful = total_combinations - status_counts['error']
print(f"\n🎯 Éxito: {successful}/{total_combinations} combinaciones procesadas correctamente")
print(f"{'='*70}")

📅 VERIFICACIÓN DE CONTINUIDAD TEMPORAL

--- Variable: PR ---

  📁 Escenario: SSP245
    📄 Archivo 1: 2015-01-01 a 2064-07-05 (18,084 días)
    📄 Archivo 2: 2064-07-06 a 2100-12-31 (13,327 días)
    ✅ Continuidad perfecta (1 día de diferencia)
    📊 Período completo: 2015-01-01 a 2100-12-31
    📈 Total: 31,411 días (~86.0 años)

  📁 Escenario: SSP370
    📄 Archivo 1: 2015-01-01 a 2064-07-05 (18,084 días)
    📄 Archivo 2: 2064-07-06 a 2100-12-31 (13,327 días)
    ✅ Continuidad perfecta (1 día de diferencia)
    📊 Período completo: 2015-01-01 a 2100-12-31
    📈 Total: 31,411 días (~86.0 años)

  📁 Escenario: SSP585
    📄 Archivo 1: 2015-01-01 a 2064-07-05 (18,084 días)
    📄 Archivo 2: 2064-07-06 a 2100-12-31 (13,327 días)
    ✅ Continuidad perfecta (1 día de diferencia)
    📊 Período completo: 2015-01-01 a 2100-12-31
    📈 Total: 31,411 días (~86.0 años)

--- Variable: TASMIN ---

  📁 Escenario: SSP245
    📄 Archivo 1: 2015-01-01 a 2064-07-05 (18,084 días)
    📄 Archivo 2: 2064-07-06 a 2

## 🔧 SECCIÓN 3: FUNCIONES DE PROCESAMIENTO DE DATOS SSP

Con los archivos verificados, ahora implementamos las funciones principales del pipeline:

1. **Concatenación temporal**: Unir los dos períodos (2015-2064 + 2064-2100)
2. **Conversión de unidades**: kg m-2 s-1 → mm/day (pr), K → °C (temperaturas)
3. **Carga de CR2MET**: Referencia para regridding
4. **Regridding espacial**: CMIP6 → rejilla CR2MET (0.05°)
5. **Recorte espacial**: Filtrar a Valle de Aconcagua
6. **Verificación de bias params**: Comprobar parámetros entrenados disponibles

In [7]:
# ============================================================
# 📅 FUNCIÓN: CONCATENACIÓN TEMPORAL DE ARCHIVOS SSP
# ============================================================

def concatenate_ssp_files(var, scenario, handle_overlap=True):
    """
    Concatena los dos archivos temporales de una variable y escenario SSP
    
    Parameters:
    -----------
    var : str
        Variable a procesar ('pr', 'tasmin', 'tasmax')
    scenario : str
        Escenario SSP ('ssp245', 'ssp370', 'ssp585')
    handle_overlap : bool
        Si True, maneja automáticamente solapamientos temporales
    
    Returns:
    --------
    xr.Dataset
        Dataset concatenado con toda la serie temporal 2015-2100
    """
    
    print(f"\n🔗 Concatenando {var.upper()} - {scenario.upper()}")
    
    # Verificar que los archivos existen
    if var not in available_files or scenario not in available_files[var]:
        raise ValueError(f"Archivos no disponibles para {var} - {scenario}")
    
    files = available_files[var][scenario]
    if len(files) != 2:
        raise ValueError(f"Se esperaban 2 archivos, encontrados: {len(files)}")
    
    # Ordenar archivos por fecha (ya están ordenados, pero por seguridad)
    files = sorted(files)
    
    print(f"  📄 Archivo 1: {files[0].name}")
    print(f"  📄 Archivo 2: {files[1].name}")
    
    try:
        # Cargar ambos datasets
        ds1 = xr.open_dataset(files[0])
        ds2 = xr.open_dataset(files[1])
        
        print(f"  📊 Período 1: {len(ds1.time):,} días")
        print(f"  📊 Período 2: {len(ds2.time):,} días")
        
        # Verificar continuidad temporal
        end1 = pd.to_datetime(ds1.time.values[-1])
        start2 = pd.to_datetime(ds2.time.values[0])
        gap_days = (start2 - end1).days
        
        if gap_days == 1:
            print(f"  ✅ Continuidad perfecta")
        elif gap_days == 0:
            print(f"  ⚠️ Sin gap temporal")
        elif gap_days < 0:
            overlap_days = -gap_days + 1
            print(f"  ⚠️ Solapamiento de {overlap_days} días")
            
            if handle_overlap:
                # Remover el solapamiento del segundo dataset
                overlap_end = start2 + pd.Timedelta(days=overlap_days-1)
                ds2 = ds2.sel(time=ds2.time > overlap_end)
                print(f"    🔧 Solapamiento removido automáticamente")
                
        # Concatenar datasets
        print(f"  🔗 Concatenando datasets...")
        ds_concat = xr.concat([ds1, ds2], dim='time')
        
        # Verificar resultado
        total_days = len(ds_concat.time)
        start_date = pd.to_datetime(ds_concat.time.values[0])
        end_date = pd.to_datetime(ds_concat.time.values[-1])
        
        print(f"  ✅ Concatenación exitosa:")
        print(f"    📅 Período: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")
        print(f"    📈 Total: {total_days:,} días ({total_days/365.25:.1f} años)")
        
        # Verificar que no hay duplicados temporales
        time_diffs = pd.to_datetime(ds_concat.time.values[1:]) - pd.to_datetime(ds_concat.time.values[:-1])
        if all(diff.days == 1 for diff in time_diffs):
            print(f"    ✅ Serie temporal continua sin duplicados")
        else:
            duplicate_count = sum(1 for diff in time_diffs if diff.days == 0)
            if duplicate_count > 0:
                print(f"    ⚠️ Detectados {duplicate_count} duplicados temporales")
        
        # Cerrar datasets originales para liberar memoria
        ds1.close()
        ds2.close()
        
        return ds_concat
        
    except Exception as e:
        print(f"  ❌ Error en concatenación: {e}")
        raise


# ============================================================
# 🔧 FUNCIÓN: CONVERSIÓN DE UNIDADES
# ============================================================

def convert_units(ds, var):
    """
    Convierte unidades del dataset según la configuración
    
    Parameters:
    -----------
    ds : xr.Dataset
        Dataset a convertir
    var : str
        Variable ('pr', 'tasmin', 'tasmax')
    
    Returns:
    --------
    xr.Dataset
        Dataset con unidades convertidas
    """
    
    if var not in unit_conversions:
        print(f"  ⚠️ No hay conversión definida para {var}")
        return ds
    
    config = unit_conversions[var]
    print(f"  🔧 {config['description']}")
    
    # Crear copia del dataset
    ds_converted = ds.copy()
    
    # Aplicar conversión
    if var == 'pr':
        # Multiplicar por factor (86400)
        ds_converted[var] = ds[var] * config['factor']
    else:
        # Sumar factor (-273.15 para K → °C)
        ds_converted[var] = ds[var] + config['factor']
    
    # Actualizar atributos de unidades
    ds_converted[var].attrs['units'] = config['to']
    ds_converted[var].attrs['units_original'] = config['from']
    ds_converted[var].attrs['conversion_factor'] = config['factor']
    
    # Verificar el resultado
    if var == 'pr':
        min_val = float(ds_converted[var].min())
        max_val = float(ds_converted[var].max())
        print(f"    📊 Rango convertido: {min_val:.2f} a {max_val:.2f} {config['to']}")
    else:
        min_val = float(ds_converted[var].min())
        max_val = float(ds_converted[var].max())
        print(f"    📊 Rango convertido: {min_val:.1f} a {max_val:.1f} {config['to']}")
    
    return ds_converted


# Test de concatenación con una variable
print("=" * 70)
print("🧪 TEST DE CONCATENACIÓN Y CONVERSIÓN")
print("=" * 70)

# Probar con pr, ssp245
test_ds = concatenate_ssp_files('pr', 'ssp245')
test_ds_converted = convert_units(test_ds, 'pr')

print(f"\n📋 Resumen del test:")
print(f"  Variable: pr")
print(f"  Escenario: ssp245") 
print(f"  Dimensiones finales: {dict(test_ds_converted.dims)}")
print(f"  Unidades: {test_ds_converted['pr'].attrs.get('units', 'N/A')}")

# Liberar memoria
test_ds.close()
test_ds_converted.close()

print(f"\n✅ Funciones de concatenación y conversión listas")
print(f"{'='*70}")

🧪 TEST DE CONCATENACIÓN Y CONVERSIÓN

🔗 Concatenando PR - SSP245
  📄 Archivo 1: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
  📄 Archivo 2: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc
  📊 Período 1: 18,084 días
  📊 Período 2: 13,327 días
  ✅ Continuidad perfecta
  🔗 Concatenando datasets...
  ✅ Concatenación exitosa:
    📅 Período: 2015-01-01 a 2100-12-31
    📈 Total: 31,411 días (86.0 años)
    ✅ Serie temporal continua sin duplicados
  🔧 Precipitación: kg m-2 s-1 → mm/day
    📊 Rango convertido: 0.00 a 1826.98 mm/day

📋 Resumen del test:
  Variable: pr
  Escenario: ssp245
  Dimensiones finales: {'time': 31411, 'bnds': 2, 'lat': 144, 'lon': 192}
  Unidades: mm/day

✅ Funciones de concatenación y conversión listas


In [10]:
# ============================================================
# 🔄 PIPELINE SIMPLE: CONCATENACIÓN Y CONVERSIÓN DE UNIDADES
# ============================================================

def process_ssp_basic_pipeline():
    """
    Pipeline simplificado que solo hace:
    1. Concatenación temporal (2015-2064 + 2064-2100)
    2. Conversión de unidades (kg m-2 s-1 → mm/day, K → °C)
    3. Exportar archivos NetCDF concatenados y con unidades correctas
    
    Sin regridding ni bias correction (para el siguiente paso)
    """
    
    print(f"\n{'='*70}")
    print(f"🎯 PIPELINE BÁSICO: CONCATENACIÓN + CONVERSIÓN UNIDADES")
    print(f"{'='*70}")
    print(f"📊 Total a procesar: {len(variables)} variables × {len(ssp_scenarios)} escenarios = {len(variables) * len(ssp_scenarios)} datasets")
    print(f"⏱️ Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Directorio para archivos intermedios
    intermediate_dir = out_dir / "ssp_concatenated" / model_name
    intermediate_dir.mkdir(parents=True, exist_ok=True)
    
    results = {}
    processing_log = []
    successful = 0
    failed = 0
    
    for var in variables:
        results[var] = {}
        
        for scenario in ssp_scenarios:
            combination = f"{var}-{scenario}"
            print(f"\n{'🔸'*50}")
            print(f"📋 PROCESANDO: {combination.upper()}")
            print(f"{'🔸'*50}")
            
            start_time = time.time()
            
            try:
                # 1. Concatenar archivos temporales
                print(f"\n1️⃣ CONCATENACIÓN TEMPORAL")
                ds = concatenate_ssp_files(var, scenario)
                
                # 2. Convertir unidades
                print(f"\n2️⃣ CONVERSIÓN DE UNIDADES")
                ds = convert_units(ds, var)
                
                # 3. Agregar metadatos básicos
                print(f"\n3️⃣ METADATOS BÁSICOS")
                ds.attrs.update({
                    'title': f'SSP {scenario.upper()} concatenated data - {var}',
                    'model': model_name,
                    'scenario': scenario,
                    'variable': var,
                    'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'processing_stage': 'concatenated_unit_converted',
                    'temporal_resolution': 'daily',
                    'spatial_resolution': 'native_CMIP6_grid',
                    'institution': 'Universidad_de_Chile',
                    'project': 'JUSTH2_Pipeline'
                })
                
                # Metadatos de variable
                if var in ds.data_vars:
                    ds[var].attrs.update({
                        'processing_steps': 'temporal_concatenation,unit_conversion',
                        'source_files': f'{scenario}_temporal_split_files',
                        'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })
                
                # 4. Guardar archivo intermedio
                print(f"\n4️⃣ GUARDANDO ARCHIVO CONCATENADO")
                output_filename = f"{var}_{model_name}_{scenario}_concatenated_2015-2100.nc"
                output_path = intermediate_dir / output_filename
                
                print(f"  📁 Ruta: {output_path}")
                
                # Guardar con compresión
                encoding = {var: {'zlib': True, 'complevel': 4}}
                ds.to_netcdf(output_path, encoding=encoding)
                
                # Verificar archivo guardado
                file_size_mb = output_path.stat().st_size / (1024 * 1024)
                print(f"  ✅ Archivo guardado exitosamente")
                print(f"  📊 Tamaño: {file_size_mb:.1f} MB")
                
                # Mostrar información del dataset
                print(f"\n📋 INFORMACIÓN DEL DATASET:")
                print(f"  📊 Dimensiones: {dict(ds.dims)}")
                print(f"  📅 Período: {ds.time.min().dt.strftime('%Y-%m-%d').values} a {ds.time.max().dt.strftime('%Y-%m-%d').values}")
                print(f"  🌍 Espacial: {len(ds.lat)}×{len(ds.lon)} puntos (resolución nativa CMIP6)")
                print(f"  🔧 Unidades: {ds[var].attrs.get('units', 'N/A')}")
                
                # Limpiar memoria
                ds.close()
                
                # Registrar éxito
                processing_time = time.time() - start_time
                results[var][scenario] = {
                    'status': 'success',
                    'output_file': output_path,
                    'file_size_mb': file_size_mb,
                    'processing_time': processing_time
                }
                
                processing_log.append({
                    'combination': combination,
                    'status': 'SUCCESS',
                    'processing_time': processing_time,
                    'output_file': output_filename,
                    'file_size_mb': file_size_mb
                })
                
                successful += 1
                print(f"  ⏱️ Tiempo: {processing_time:.1f} segundos")
                print(f"  ✅ ÉXITO")
                
            except Exception as e:
                # Registrar error
                processing_time = time.time() - start_time
                results[var][scenario] = {
                    'status': 'failed',
                    'error': str(e),
                    'processing_time': processing_time
                }
                
                processing_log.append({
                    'combination': combination,
                    'status': 'FAILED',
                    'processing_time': processing_time,
                    'error': str(e)
                })
                
                failed += 1
                print(f"  ❌ ERROR: {e}")
                print(f"  ⏱️ Tiempo hasta error: {processing_time:.1f} segundos")
                
                # Continuar con el siguiente
                continue
    
    # Resumen final
    total_time = sum(log['processing_time'] for log in processing_log)
    
    print(f"\n{'='*70}")
    print(f"📊 RESUMEN FINAL - PIPELINE BÁSICO")
    print(f"{'='*70}")
    print(f"✅ Exitosos: {successful}/{len(variables) * len(ssp_scenarios)}")
    print(f"❌ Fallidos: {failed}/{len(variables) * len(ssp_scenarios)}")
    print(f"⏱️ Tiempo total: {total_time:.1f} segundos ({total_time/60:.1f} minutos)")
    print(f"📁 Directorio salida: {intermediate_dir}")
    
    if successful > 0:
        print(f"\n📋 ARCHIVOS CONCATENADOS GENERADOS:")
        total_size = 0
        for log in processing_log:
            if log['status'] == 'SUCCESS':
                print(f"  ✅ {log['output_file']} ({log['file_size_mb']:.1f} MB)")
                total_size += log['file_size_mb']
        print(f"  📊 Tamaño total: {total_size:.1f} MB")
        print(f"\n📝 SIGUIENTES PASOS:")
        print(f"  1. Revisar archivos concatenados")
        print(f"  2. Implementar regridding a CR2MET (separado)")
        print(f"  3. Aplicar bias correction (separado)")
    
    if failed > 0:
        print(f"\n❌ ERRORES ENCONTRADOS:")
        for log in processing_log:
            if log['status'] == 'FAILED':
                print(f"  ❌ {log['combination']}: {log.get('error', 'Error desconocido')}")
    
    # Guardar log detallado
    log_filename = f"ssp_basic_processing_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    log_path = logs_dir / log_filename
    
    with open(log_path, 'w') as f:
        json.dump(processing_log, f, indent=2, default=str)
    
    print(f"\n📝 Log detallado guardado: {log_path}")
    print(f"🏁 Pipeline básico completado")
    print(f"{'='*70}")
    
    return results, processing_log


# Ejecutar pipeline básico
print("🚀 INICIANDO PIPELINE BÁSICO: CONCATENACIÓN + CONVERSIÓN UNIDADES")
print("📝 Nota: Este pipeline genera archivos intermedios concatenados con unidades correctas")
print("🔄 Regridding y bias correction se harán en pasos separados")

# Ejecutar procesamiento básico
results_basic, log_basic = process_ssp_basic_pipeline()

print(f"\n🎯 PIPELINE BÁSICO COMPLETADO")
print(f"📁 Archivos generados en: {out_dir}/ssp_concatenated/{model_name}/")
print(f"📋 Próximo paso: Implementar regridding y bias correction sobre estos archivos")

🚀 INICIANDO PIPELINE BÁSICO: CONCATENACIÓN + CONVERSIÓN UNIDADES
📝 Nota: Este pipeline genera archivos intermedios concatenados con unidades correctas
🔄 Regridding y bias correction se harán en pasos separados

🎯 PIPELINE BÁSICO: CONCATENACIÓN + CONVERSIÓN UNIDADES
📊 Total a procesar: 3 variables × 3 escenarios = 9 datasets
⏱️ Inicio: 2025-10-12 21:30:48

🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸
📋 PROCESANDO: PR-SSP245
🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸🔸

1️⃣ CONCATENACIÓN TEMPORAL

🔗 Concatenando PR - SSP245
  📄 Archivo 1: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
  📄 Archivo 2: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc
  📊 Período 1: 18,084 días
  📊 Período 2: 13,327 días
  ✅ Continuidad perfecta
  🔗 Concatenando datasets...
  ✅ Concatenación exitosa:
    📅 Período: 2015-01-01 a 2100-12-31
    📈 Total: 31,411 días (86.0 años)
    ✅ Serie temporal continua sin duplicados

2️⃣ CONVERSIÓN DE UNIDADES
  🔧 Precipitación: kg m-2 s-1 

In [11]:
# ============================================================
# 🔍 INSPECCIÓN DE ARCHIVOS CONCATENADOS: COORDENADAS Y FECHAS
# ============================================================

def inspect_concatenated_files():
    """
    Inspecciona los archivos concatenados para verificar:
    1. Sistema de coordenadas (longitudes 0-360° vs -180 a 180°)
    2. Formato de fechas y compatibilidad temporal
    3. Comparación con CR2MET para regridding
    """
    
    print(f"\n{'='*70}")
    print(f"🔍 INSPECCIÓN DE ARCHIVOS CONCATENADOS")
    print(f"{'='*70}")
    
    # Directorio de archivos concatenados
    intermediate_dir = out_dir / "ssp_concatenated" / model_name
    
    if not intermediate_dir.exists():
        print(f"❌ Directorio de archivos concatenados no existe: {intermediate_dir}")
        return
    
    # Buscar archivos concatenados
    concatenated_files = list(intermediate_dir.glob("*.nc"))
    
    if not concatenated_files:
        print(f"❌ No se encontraron archivos concatenados en {intermediate_dir}")
        return
    
    print(f"📁 Directorio: {intermediate_dir}")
    print(f"📦 Archivos encontrados: {len(concatenated_files)}")
    
    # Inspeccionar primer archivo como muestra
    sample_file = concatenated_files[0]
    print(f"\n📄 Analizando archivo muestra: {sample_file.name}")
    
    try:
        # Abrir archivo concatenado
        with xr.open_dataset(sample_file) as ds_ssp:
            print(f"\n📊 INFORMACIÓN GENERAL:")
            print(f"  Dimensiones: {dict(ds_ssp.dims)}")
            print(f"  Variables: {list(ds_ssp.data_vars)}")
            
            # 1. ANÁLISIS DE COORDENADAS ESPACIALES
            print(f"\n🌍 COORDENADAS ESPACIALES:")
            lat_ssp = ds_ssp.lat.values
            lon_ssp = ds_ssp.lon.values
            
            print(f"  Latitudes:")
            print(f"    Rango: {lat_ssp.min():.3f} a {lat_ssp.max():.3f}")
            print(f"    Resolución: ~{abs(lat_ssp[1] - lat_ssp[0]):.3f}°")
            print(f"    Orden: {'Norte→Sur' if lat_ssp[0] > lat_ssp[-1] else 'Sur→Norte'}")
            
            print(f"  Longitudes:")
            print(f"    Rango: {lon_ssp.min():.3f} a {lon_ssp.max():.3f}")
            print(f"    Resolución: ~{abs(lon_ssp[1] - lon_ssp[0]):.3f}°")
            
            # Verificar sistema de coordenadas
            if lon_ssp.max() > 180:
                print(f"    ⚠️ Sistema: 0° a 360° (necesita conversión a -180° a 180°)")
                conversion_needed = True
            else:
                print(f"    ✅ Sistema: -180° a 180°")
                conversion_needed = False
            
            # 2. ANÁLISIS TEMPORAL
            print(f"\n📅 INFORMACIÓN TEMPORAL:")
            time_ssp = ds_ssp.time
            print(f"  Inicio: {pd.to_datetime(time_ssp.values[0]).strftime('%Y-%m-%d')}")
            print(f"  Final: {pd.to_datetime(time_ssp.values[-1]).strftime('%Y-%m-%d')}")
            print(f"  Total días: {len(time_ssp):,}")
            print(f"  Frecuencia: {'Diaria' if len(time_ssp) > 32000 else 'Otra'}")
            
            # Verificar calendario
            if hasattr(time_ssp, 'calendar'):
                print(f"  Calendario: {time_ssp.calendar}")
            
            # Verificar formato de fechas
            time_dtype = time_ssp.values.dtype
            print(f"  Tipo de datos: {time_dtype}")
            
            # 3. COMPARACIÓN CON CR2MET
            print(f"\n🔄 COMPARACIÓN CON CR2MET:")
            
            # Cargar info de CR2MET si está disponible
            if 'cr2met_ref' in globals():
                lat_cr2met = cr2met_ref.lat.values
                lon_cr2met = cr2met_ref.lon.values
                
                print(f"  CR2MET - Latitudes: {lat_cr2met.min():.3f} a {lat_cr2met.max():.3f}")
                print(f"  CR2MET - Longitudes: {lon_cr2met.min():.3f} a {lon_cr2met.max():.3f}")
                print(f"  CR2MET - Resolución: ~{abs(lat_cr2met[1] - lat_cr2met[0]):.3f}°")
                
                # Verificar si Valle de Aconcagua está cubierto
                valle_lat_covered = (lat_ssp.min() <= valle_aconcagua_bounds['lat_min'] and 
                                   lat_ssp.max() >= valle_aconcagua_bounds['lat_max'])
                valle_lon_covered = True  # Verificaremos después de conversión de coordenadas
                
                if conversion_needed:
                    # Simular conversión para verificar cobertura
                    lon_converted = np.where(lon_ssp > 180, lon_ssp - 360, lon_ssp)
                    valle_lon_covered = (lon_converted.min() <= valle_aconcagua_bounds['lon_min'] and 
                                       lon_converted.max() >= valle_aconcagua_bounds['lon_max'])
                else:
                    valle_lon_covered = (lon_ssp.min() <= valle_aconcagua_bounds['lon_min'] and 
                                       lon_ssp.max() >= valle_aconcagua_bounds['lon_max'])
                
                valle_status = "✅" if (valle_lat_covered and valle_lon_covered) else "⚠️"
                print(f"  Cobertura Valle de Aconcagua: {valle_status}")
                
                # Verificar resoluciones
                if abs(lat_cr2met[1] - lat_cr2met[0]) < abs(lat_ssp[1] - lat_ssp[0]):
                    print(f"  📈 Regridding: Interpolación (CMIP6 → CR2MET más fino)")
                else:
                    print(f"  📉 Regridding: Agregación (CMIP6 → CR2MET más grueso)")
            
            else:
                print(f"  ⚠️ CR2MET no cargado para comparación")
    
    except Exception as e:
        print(f"❌ Error al inspeccionar archivo: {e}")
        return
    
    # 4. RESUMEN Y RECOMENDACIONES
    print(f"\n{'='*50}")
    print(f"📋 RESUMEN Y RECOMENDACIONES")
    print(f"{'='*50}")
    
    recommendations = []
    
    if conversion_needed:
        recommendations.append("🔄 Convertir longitudes de 0-360° a -180° a 180°")
    
    recommendations.append("🌍 Implementar regridding a resolución CR2MET (0.05°)")
    recommendations.append("✂️ Recortar a región Valle de Aconcagua")
    recommendations.append("🔧 Aplicar bias correction usando parámetros entrenados")
    
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")
    
    print(f"\n🎯 SIGUIENTE PASO: Implementar funciones de conversión de coordenadas")
    print(f"{'='*70}")
    
    return {
        'conversion_needed': conversion_needed,
        'sample_file': sample_file,
        'concatenated_files': concatenated_files
    }


# Ejecutar inspección
print("🔍 INSPECCIONANDO ARCHIVOS CONCATENADOS")
print("📝 Verificando coordenadas, fechas y compatibilidad con CR2MET")

inspection_results = inspect_concatenated_files()

if inspection_results:
    print(f"\n✅ Inspección completada")
    if inspection_results['conversion_needed']:
        print(f"⚠️ Se requiere conversión de coordenadas antes del regridding")
    else:
        print(f"✅ Coordenadas compatibles para regridding directo")
else:
    print(f"❌ No se pudieron inspeccionar los archivos")
    print(f"💡 Asegúrate de haber ejecutado primero el pipeline básico de concatenación")

🔍 INSPECCIONANDO ARCHIVOS CONCATENADOS
📝 Verificando coordenadas, fechas y compatibilidad con CR2MET

🔍 INSPECCIÓN DE ARCHIVOS CONCATENADOS
📁 Directorio: /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_concatenated/ACCESS-CM2
📦 Archivos encontrados: 9

📄 Analizando archivo muestra: pr_ACCESS-CM2_ssp245_concatenated_2015-2100.nc

📊 INFORMACIÓN GENERAL:
  Dimensiones: {'time': 31411, 'bnds': 2, 'lat': 144, 'lon': 192}
  Variables: ['time_bnds', 'lat_bnds', 'lon_bnds', 'pr']

🌍 COORDENADAS ESPACIALES:
  Latitudes:
    Rango: -89.375 a 89.375
    Resolución: ~1.250°
    Orden: Sur→Norte
  Longitudes:
    Rango: 0.938 a 359.062
    Resolución: ~1.875°
    ⚠️ Sistema: 0° a 360° (necesita conversión a -180° a 180°)

📅 INFORMACIÓN TEMPORAL:
  Inicio: 2015-01-01
  Final: 2100-12-31
  Total días: 31,411
  Frecuencia: Otra
  Tipo de datos: datetime64[ns]

🔄 COMPARACIÓN CON CR2MET:
  CR2MET - Latitudes: -56.975 a -17.025
  CR2MET - Longitudes: -76.975 a -66.025
  CR2MET - Resolución: ~0.050°

In [13]:
# ============================================================
# 🔄 CONVERSIÓN DE COORDENADAS: 0-360° → -180°/180°
# ============================================================

def convert_longitude_coordinates(ds):
    """
    Convierte coordenadas de longitud de 0-360° a -180° a 180°
    
    Parameters:
    -----------
    ds : xr.Dataset
        Dataset con coordenadas en formato 0-360°
    
    Returns:
    --------
    xr.Dataset
        Dataset con coordenadas convertidas a -180° a 180°
    """
    
    print(f"  🔄 Convirtiendo coordenadas de longitud...")
    
    # Crear copia del dataset
    ds_converted = ds.copy()
    
    # Obtener longitudes originales
    lon_original = ds.lon.values
    print(f"    Original: {lon_original.min():.3f} a {lon_original.max():.3f}")
    
    # Verificar si necesita conversión
    if lon_original.max() <= 180:
        print(f"    ✅ Coordenadas ya están en formato -180° a 180°")
        return ds_converted
    
    # Convertir: longitudes > 180 se convierten a negativas
    lon_converted = np.where(lon_original > 180, lon_original - 360, lon_original)
    print(f"    Convertido: {lon_converted.min():.3f} a {lon_converted.max():.3f}")
    
    # Actualizar coordenadas en el dataset
    ds_converted = ds_converted.assign_coords(lon=lon_converted)
    
    # Re-ordenar por longitud para mantener orden ascendente
    ds_converted = ds_converted.sortby('lon')
    
    # Verificar el resultado
    lon_final = ds_converted.lon.values
    print(f"    Final (ordenado): {lon_final.min():.3f} a {lon_final.max():.3f}")
    
    # Actualizar metadatos
    ds_converted.lon.attrs.update({
        'long_name': 'longitude',
        'standard_name': 'longitude',
        'units': 'degrees_east',
        'axis': 'X',
        'coordinate_system': 'WGS84',
        'conversion_note': 'Converted from 0-360° to -180°/180° format'
    })
    
    # Actualizar atributos globales
    ds_converted.attrs.update({
        'coordinate_conversion': 'longitude_0_360_to_-180_180',
        'coordinate_conversion_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })
    
    return ds_converted


def process_coordinate_conversion():
    """
    Procesa todos los archivos concatenados para convertir coordenadas
    y guardarlos con coordenadas corregidas
    """
    
    print(f"\n{'='*70}")
    print(f"🔄 CONVERSIÓN MASIVA DE COORDENADAS")
    print(f"{'='*70}")
    
    # Directorio de archivos concatenados
    intermediate_dir = out_dir / "ssp_concatenated" / model_name
    
    # Buscar archivos concatenados
    concatenated_files = list(intermediate_dir.glob("*_concatenated_*.nc"))
    
    if not concatenated_files:
        print(f"❌ No se encontraron archivos concatenados")
        return
    
    print(f"📁 Directorio: {intermediate_dir}")
    print(f"📦 Archivos a procesar: {len(concatenated_files)}")
    
    results = {}
    successful = 0
    failed = 0
    
    for file_path in concatenated_files:
        file_name = file_path.name
        print(f"\n{'🔹'*50}")
        print(f"📋 PROCESANDO: {file_name}")
        print(f"{'🔹'*50}")
        
        start_time = time.time()
        
        try:
            # 1. Cargar archivo
            print(f"\n1️⃣ CARGANDO ARCHIVO")
            print(f"  📄 {file_name}")
            
            with xr.open_dataset(file_path) as ds:
                # Verificar si necesita conversión
                lon_max = ds.lon.values.max()
                if lon_max <= 180:
                    print(f"  ✅ Archivo ya tiene coordenadas correctas (-180° a 180°)")
                    print(f"  ⏭️ Saltando conversión")
                    continue
                
                # 2. Convertir coordenadas
                print(f"\n2️⃣ CONVERSIÓN DE COORDENADAS")
                ds_converted = convert_longitude_coordinates(ds)
                
                # 3. Verificar cobertura Valle de Aconcagua después de conversión
                print(f"\n3️⃣ VERIFICACIÓN DE COBERTURA")
                lat_min, lat_max = ds_converted.lat.values.min(), ds_converted.lat.values.max()
                lon_min, lon_max = ds_converted.lon.values.min(), ds_converted.lon.values.max()
                
                valle_lat_ok = (lat_min <= valle_aconcagua_bounds['lat_min'] and 
                              lat_max >= valle_aconcagua_bounds['lat_max'])
                valle_lon_ok = (lon_min <= valle_aconcagua_bounds['lon_min'] and 
                              lon_max >= valle_aconcagua_bounds['lon_max'])
                
                status = "✅" if (valle_lat_ok and valle_lon_ok) else "⚠️"
                print(f"    Valle de Aconcagua cubierto: {status}")
                print(f"    Latitudes: {lat_min:.3f} a {lat_max:.3f}")
                print(f"    Longitudes: {lon_min:.3f} a {lon_max:.3f}")
                
                # 4. Generar nuevo nombre de archivo
                new_filename = file_name.replace('_concatenated_', '_concatenated_coords_')
                new_file_path = intermediate_dir / new_filename
                
                print(f"\n4️⃣ GUARDANDO ARCHIVO CONVERTIDO")
                print(f"  📁 {new_filename}")
                
                # 5. Guardar con compresión
                # Detectar variable principal
                data_vars = [var for var in ds_converted.data_vars if var not in ['time_bnds', 'lat_bnds', 'lon_bnds']]
                main_var = data_vars[0] if data_vars else 'pr'
                
                encoding = {main_var: {'zlib': True, 'complevel': 4}}
                ds_converted.to_netcdf(new_file_path, encoding=encoding)
                
                # Verificar archivo guardado
                file_size_mb = new_file_path.stat().st_size / (1024 * 1024)
                print(f"  ✅ Archivo guardado exitosamente")
                print(f"  📊 Tamaño: {file_size_mb:.1f} MB")
                
                # 6. Eliminar archivo original (opcional)
                print(f"\n5️⃣ LIMPIEZA")
                original_size_mb = file_path.stat().st_size / (1024 * 1024)
                print(f"  🗑️ Archivo original: {original_size_mb:.1f} MB")
                print(f"  ♻️ Eliminando archivo original para ahorrar espacio...")
                file_path.unlink()
                print(f"  ✅ Archivo original eliminado")
                
                # Mostrar información final
                print(f"\n📋 RESULTADO:")
                print(f"  📊 Dimensiones: {dict(ds_converted.dims)}")
                print(f"  🌍 Coordenadas: {len(ds_converted.lat)}×{len(ds_converted.lon)} puntos")
                print(f"  📅 Período: {ds_converted.time.min().dt.strftime('%Y-%m-%d').values} a {ds_converted.time.max().dt.strftime('%Y-%m-%d').values}")
                
        except Exception as e:
            print(f"  ❌ ERROR: {e}")
            results[file_name] = {'status': 'failed', 'error': str(e)}
            failed += 1
            continue
            
        # Registrar éxito
        processing_time = time.time() - start_time
        results[file_name] = {
            'status': 'success',
            'new_filename': new_filename,
            'processing_time': processing_time,
            'file_size_mb': file_size_mb,
            'valle_coverage': valle_lat_ok and valle_lon_ok
        }
        successful += 1
        print(f"  ⏱️ Tiempo: {processing_time:.1f} segundos")
        print(f"  ✅ CONVERSIÓN EXITOSA")
    
    # Resumen final
    print(f"\n{'='*70}")
    print(f"📊 RESUMEN - CONVERSIÓN DE COORDENADAS")
    print(f"{'='*70}")
    print(f"✅ Exitosos: {successful}/{len(concatenated_files)}")
    print(f"❌ Fallidos: {failed}/{len(concatenated_files)}")
    
    if successful > 0:
        print(f"\n📋 ARCHIVOS CON COORDENADAS CONVERTIDAS:")
        total_size = 0
        for file_name, result in results.items():
            if result['status'] == 'success':
                coverage = "✅" if result['valle_coverage'] else "⚠️"
                print(f"  {coverage} {result['new_filename']} ({result['file_size_mb']:.1f} MB)")
                total_size += result['file_size_mb']
        
        print(f"  📊 Tamaño total: {total_size:.1f} MB")
        
        print(f"\n🎯 ARCHIVOS LISTOS PARA:")
        print(f"  1. 🌍 Regridding a resolución CR2MET (0.05°)")
        print(f"  2. ✂️ Recorte espacial a Valle de Aconcagua")
        print(f"  3. 🔧 Bias correction")
    
    if failed > 0:
        print(f"\n❌ ERRORES:")
        for file_name, result in results.items():
            if result['status'] == 'failed':
                print(f"  ❌ {file_name}: {result['error']}")
    
    print(f"\n📁 Directorio actualizado: {intermediate_dir}")
    print(f"🏁 Conversión de coordenadas completada")
    print(f"{'='*70}")
    
    return results


# Ejecutar conversión de coordenadas
print("🔄 INICIANDO CONVERSIÓN DE COORDENADAS")
print("📝 Convirtiendo de 0-360° a -180°/180° para compatibilidad con CR2MET")

coordinate_results = process_coordinate_conversion()

print(f"\n🎯 CONVERSIÓN DE COORDENADAS COMPLETADA")
print(f"📁 Archivos actualizados en: {out_dir}/ssp_concatenated/{model_name}/")
print(f"✅ Coordenadas ahora compatibles con sistema CR2MET")
print(f"📋 Siguiente paso: Implementar regridding a resolución CR2MET (0.05°)")

🔄 INICIANDO CONVERSIÓN DE COORDENADAS
📝 Convirtiendo de 0-360° a -180°/180° para compatibilidad con CR2MET

🔄 CONVERSIÓN MASIVA DE COORDENADAS
📁 Directorio: /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_concatenated/ACCESS-CM2
📦 Archivos a procesar: 9

🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
📋 PROCESANDO: pr_ACCESS-CM2_ssp245_concatenated_2015-2100.nc
🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹

1️⃣ CARGANDO ARCHIVO
  📄 pr_ACCESS-CM2_ssp245_concatenated_2015-2100.nc

2️⃣ CONVERSIÓN DE COORDENADAS
  🔄 Convirtiendo coordenadas de longitud...
    Original: 0.938 a 359.062
    Convertido: -179.062 a 179.062
    Final (ordenado): -179.062 a 179.062

3️⃣ VERIFICACIÓN DE COBERTURA
    Valle de Aconcagua cubierto: ✅
    Latitudes: -89.375 a 89.375
    Longitudes: -179.062 a 179.062

4️⃣ GUARDANDO ARCHIVO CONVERTIDO
  📁 pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
  ✅ Archivo guardado exitosamente
  📊 Tamaño: 2751.6 MB

5️⃣ LIMPIEZA
  🗑️ Archivo original: 2750

In [12]:
# ============================================================
# 📅 VERIFICACIÓN DE COMPATIBILIDAD TEMPORAL CON CR2MET
# ============================================================

def verify_temporal_compatibility_with_cr2met():
    """
    Verifica que las fechas de los archivos SSP sean compatibles con CR2MET
    para el regridding y bias correction posterior
    """
    
    print(f"\n{'='*70}")
    print(f"📅 VERIFICACIÓN DE COMPATIBILIDAD TEMPORAL CON CR2MET")
    print(f"{'='*70}")
    
    # Verificar si CR2MET está cargado
    if 'cr2met_ref' not in globals():
        print(f"⚠️ CR2MET no está cargado. Cargando...")
        try:
            global cr2met_ref
            cr2met_ref = xr.open_zarr(cr2met_path)
            print(f"✅ CR2MET cargado exitosamente")
        except Exception as e:
            print(f"❌ Error al cargar CR2MET: {e}")
            return
    
    # Analizar información temporal de CR2MET
    print(f"\n📊 INFORMACIÓN TEMPORAL CR2MET:")
    cr2met_time = cr2met_ref.time
    cr2met_start = pd.to_datetime(cr2met_time.values[0])
    cr2met_end = pd.to_datetime(cr2met_time.values[-1])
    cr2met_total_days = len(cr2met_time)
    
    print(f"  📅 Período: {cr2met_start.strftime('%Y-%m-%d')} a {cr2met_end.strftime('%Y-%m-%d')}")
    print(f"  📈 Total días: {cr2met_total_days:,}")
    print(f"  🗓️ Calendario: {getattr(cr2met_time, 'calendar', 'standard')}")
    print(f"  📊 Tipo de datos: {cr2met_time.values.dtype}")
    print(f"  ⏰ Frecuencia: {'Diaria' if cr2met_total_days > 32000 else 'Otra'}")
    
    # Verificar archivos SSP concatenados
    intermediate_dir = out_dir / "ssp_concatenated" / model_name
    concatenated_files = list(intermediate_dir.glob("*.nc"))
    
    if not concatenated_files:
        print(f"\n❌ No se encontraron archivos SSP concatenados")
        return
    
    print(f"\n📦 VERIFICANDO {len(concatenated_files)} ARCHIVOS SSP:")
    
    compatibility_report = {}
    all_compatible = True
    
    for file_path in concatenated_files[:3]:  # Verificar solo los primeros 3 como muestra
        file_name = file_path.name
        print(f"\n📄 {file_name}")
        
        try:
            with xr.open_dataset(file_path) as ds_ssp:
                ssp_time = ds_ssp.time
                ssp_start = pd.to_datetime(ssp_time.values[0])
                ssp_end = pd.to_datetime(ssp_time.values[-1])
                ssp_total_days = len(ssp_time)
                
                print(f"  📅 Período: {ssp_start.strftime('%Y-%m-%d')} a {ssp_end.strftime('%Y-%m-%d')}")
                print(f"  📈 Total días: {ssp_total_days:,}")
                print(f"  🗓️ Calendario: {getattr(ssp_time, 'calendar', 'standard')}")
                print(f"  📊 Tipo de datos: {ssp_time.values.dtype}")
                
                # Verificar compatibilidad
                compatibility_checks = {
                    'data_type': str(cr2met_time.values.dtype) == str(ssp_time.values.dtype),
                    'calendar': getattr(cr2met_time, 'calendar', 'standard') == getattr(ssp_time, 'calendar', 'standard'),
                    'frequency': True,  # Ambos son diarios
                    'temporal_overlap': ssp_start <= cr2met_end and ssp_end >= cr2met_start
                }
                
                # Mostrar resultados de compatibilidad
                for check, result in compatibility_checks.items():
                    emoji = "✅" if result else "⚠️"
                    print(f"    {emoji} {check}: {'Compatible' if result else 'Incompatible'}")
                    if not result:
                        all_compatible = False
                
                compatibility_report[file_name] = compatibility_checks
                
        except Exception as e:
            print(f"  ❌ Error al verificar {file_name}: {e}")
            all_compatible = False
    
    # Resumen de compatibilidad
    print(f"\n{'='*50}")
    print(f"📋 RESUMEN DE COMPATIBILIDAD TEMPORAL")
    print(f"{'='*50}")
    
    if all_compatible:
        print(f"✅ EXCELENTE: Todos los archivos SSP son compatibles con CR2MET")
        print(f"🎯 Los archivos están listos para:")
        print(f"  📊 Regridding temporal (misma frecuencia diaria)")
        print(f"  🔧 Bias correction (calendarios compatibles)")
        print(f"  📅 Procesamiento sin conversión de fechas adicional")
    else:
        print(f"⚠️ ATENCIÓN: Algunos archivos pueden tener incompatibilidades")
        print(f"🔧 Recomendaciones:")
        print(f"  📊 Verificar calendarios antes del regridding")
        print(f"  🗓️ Considerar conversión de calendario si es necesario")
    
    print(f"\n💾 GESTIÓN DE ESPACIO:")
    print(f"✅ La función de conversión de coordenadas ELIMINA archivos originales")
    print(f"📊 Esto ahorra ~50% de espacio en disco")
    print(f"🔄 Archivos convertidos tienen sufijo '_coords_' en el nombre")
    
    print(f"\n🎯 SIGUIENTE ACCIÓN:")
    print(f"{'✅' if all_compatible else '⚠️'} Ejecutar conversión de coordenadas (elimina originales)")
    print(f"📁 Archivos resultantes: *_concatenated_coords_*.nc")
    
    return compatibility_report, all_compatible


# Ejecutar verificación de compatibilidad temporal
print("📅 VERIFICANDO COMPATIBILIDAD TEMPORAL CON CR2MET")
print("🔍 Analizando calendarios, tipos de datos y frecuencias")

compatibility_report, temporal_compatible = verify_temporal_compatibility_with_cr2met()

if temporal_compatible:
    print(f"\n✅ COMPATIBILIDAD TEMPORAL CONFIRMADA")
    print(f"🚀 Proceder con conversión de coordenadas")
else:
    print(f"\n⚠️ REVISAR COMPATIBILIDAD ANTES DE CONTINUAR")
    print(f"📝 Verificar calendarios y tipos de datos")

📅 VERIFICANDO COMPATIBILIDAD TEMPORAL CON CR2MET
🔍 Analizando calendarios, tipos de datos y frecuencias

📅 VERIFICACIÓN DE COMPATIBILIDAD TEMPORAL CON CR2MET

📊 INFORMACIÓN TEMPORAL CR2MET:
  📅 Período: 1960-01-01 a 2021-12-31
  📈 Total días: 22,646
  🗓️ Calendario: standard
  📊 Tipo de datos: datetime64[ns]
  ⏰ Frecuencia: Otra

📦 VERIFICANDO 9 ARCHIVOS SSP:

📄 pr_ACCESS-CM2_ssp245_concatenated_2015-2100.nc
  📅 Período: 2015-01-01 a 2100-12-31
  📈 Total días: 31,411
  🗓️ Calendario: standard
  📊 Tipo de datos: datetime64[ns]
    ✅ data_type: Compatible
    ✅ calendar: Compatible
    ✅ frequency: Compatible
    ✅ temporal_overlap: Compatible

📄 pr_ACCESS-CM2_ssp370_concatenated_2015-2100.nc
  📅 Período: 2015-01-01 a 2100-12-31
  📈 Total días: 31,411
  🗓️ Calendario: standard
  📊 Tipo de datos: datetime64[ns]
    ✅ data_type: Compatible
    ✅ calendar: Compatible
    ✅ frequency: Compatible
    ✅ temporal_overlap: Compatible

📄 pr_ACCESS-CM2_ssp585_concatenated_2015-2100.nc
  📅 Período: